# Titanic — five iterations to 0.81100

| Version | Idea | Score |
|---|---|---|
| V1 | 4 raw features | 0.77511 |
| V2 | Title + family size | 0.78708 |
| V3 | Deck + ensemble | 0.76794 |
| V4 | Pruned + regularised | 0.77272 |
| **V5** | **Family-group survival** | **0.81100** |

Run top to bottom — later cells reuse earlier variables.


In [1]:
import pandas as pd
import numpy as np

## Load the data


In [2]:
import os

DATA = "/kaggle/input/titanic" if os.path.isdir("/kaggle/input/titanic") else "."

train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
print(train.head(5))


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [3]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print("\nNumber of survival rate in train")
print(train['Survived'].value_counts(normalize=True))

print("\n Missing values in the train:")
print(train.isnull().sum()[train.isnull().sum()>0])

Train Shape: (891, 12)
Test Shape: (418, 11)

Number of survival rate in train
0    0.616162
1    0.383838
Name: Survived, dtype: float64

 Missing values in the train:
Age         177
Cabin       687
Embarked      2
dtype: int64


In [4]:
print("-- Survival by Sex --")
print(train.groupby('Sex')['Survived'].mean())

print("-- Survival by Class --")
print(train.groupby('Pclass')['Survived'].mean())

print("\n Survival by class and Sex")
print(train.pivot_table(values='Survived',index='Pclass',columns='Sex'))

-- Survival by Sex --
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
-- Survival by Class --
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

 Survival by class and Sex
Sex       female      male
Pclass                    
1       0.968085  0.368852
2       0.921053  0.157407
3       0.500000  0.135447


## Version 1 — Baseline · 0.77511

`Pclass`, `Sex`, `SibSp`, `Parch`. Learns *female lives, male dies*.


In [5]:
features = ["Pclass", 'Sex', 'SibSp', 'Parch']

x = pd.get_dummies(train[features],drop_first=True)
y = train['Survived']

x_test= pd.get_dummies(test[features],drop_first=True)

print("Features Datafram (X):")
print(x.head())

Features Datafram (X):
   Pclass  SibSp  Parch  Sex_male
0       3      1      0         1
1       1      1      0         0
2       3      0      0         0
3       1      1      0         0
4       3      0      0         1


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train,X_val,y_train,y_val = train_test_split(x,y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train,y_train)

val_predictions= model.predict(X_val)
accuracy=accuracy_score(y_val,val_predictions)

print(f"Validatoin Accuracy: {accuracy * 100:.2f}%")

Validatoin Accuracy: 78.77%


In [7]:
# final_predictions=model.predict(x_test)

# submission= pd.DataFrame({
#     'PassengerId': test['PassengerId'],
#     'Survived': final_predictions
# })

# submission.to_csv('submission.csv',index=False)

# print("Submission file successfully created")
# print(f"file Shape: {submission.shape}")
# print("\n First 5 row of your submission are:")
# print(submission.head())

## Version 2 — Title + family · 0.78708

`Title` from `Name` (a boy under ~13 is "Master", and `Master` survived at 57.5% vs 15.7% for
adult `Mr`), plus `FamilySize`, `IsAlone`, and age imputed per title.


In [8]:
# Extract the title (e.g., Mr, Mrs, Miss, Master) from the Name column
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Look at the title distribution and survival rates
print("--- Title Counts in Train ---")
print(train['Title'].value_counts())

print("\n--- Survival Rate by Title ---")
print(train.groupby('Title')['Survived'].mean().sort_values(ascending=False))


--- Title Counts in Train ---
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Capt          1
Jonkheer      1
Countess      1
Mme           1
Don           1
Sir           1
Ms            1
Lady          1
Name: Title, dtype: int64

--- Survival Rate by Title ---
Title
Sir         1.000000
Lady        1.000000
Ms          1.000000
Countess    1.000000
Mme         1.000000
Mlle        1.000000
Mrs         0.792000
Miss        0.697802
Master      0.575000
Major       0.500000
Col         0.500000
Dr          0.428571
Mr          0.156673
Rev         0.000000
Jonkheer    0.000000
Don         0.000000
Capt        0.000000
Name: Survived, dtype: float64


In [9]:
# ============================================================
# STEP 1: TITLE GROUPING & FEATURE ENGINEERING
# ============================================================

# 1. Simplify Titles into 5 core sociological groups:
#    - 'Mr': Adult males (low survival ~15.7%)
#    - 'Miss': Unmarried females / young girls (~70% survival)
#    - 'Mrs': Married females (~79.2% survival)
#    - 'Master': Young boys under ~13 (high survival ~57.5% due to child priority)
#    - 'Rare': Officers, clergy, nobility (Dr, Rev, Col, Countess, Sir, etc.)
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Don': 'Rare', 'Rev': 'Rare', 'Dr': 'Rare', 'Major': 'Rare',
    'Lady': 'Rare', 'Sir': 'Rare', 'Col': 'Rare', 'Capt': 'Rare',
    'Countess': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
}

train['Title'] = train['Title'].map(title_mapping).fillna('Rare')
test['Title'] = test['Title'].map(title_mapping).fillna('Rare')

# 2. Family Features:
#    - FamilySize: Total traveling party (SibSp + Parch + passenger themselves)
#    - IsAlone: Binary flag (1 = solo traveler, 0 = with family)
#      Solo travelers and very large families had lower survival rates than small families (2-4).
for df in [train, test]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 3. Smart Age Imputation:
#    - ~20% of passengers have missing Age.
#    - A boy with title 'Master' should NOT be assigned median age 28 (an adult).
#    - We impute missing age using the MEDIAN age of each Title group:
#      * Master median: ~4 years
#      * Miss median: ~21 years
#      * Mr median: ~30 years
#      * Mrs median: ~35 years
train['Age'] = train.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
test['Age'] = test.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

# 4. Fill Remaining Missing Values:
#    - Fare: 1 missing row in test set (Passenger 1044, 3rd class) -> fill with median fare.
#    - Embarked: 2 missing in train -> fill with most common port 'S' (Southampton).
test['Fare'] = test['Fare'].fillna(train['Fare'].median())
train['Embarked'] = train['Embarked'].fillna('S')
test['Embarked'] = test['Embarked'].fillna('S')

# Verify: Confirm 0 missing values remain in our working features
print("Missing values remaining in train:")
print(train[['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']].isnull().sum())
print("\nMissing values remaining in test:")
print(test[['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']].isnull().sum())


Missing values remaining in train:
Pclass        0
Sex           0
Age           0
Fare          0
Embarked      0
Title         0
FamilySize    0
IsAlone       0
dtype: int64

Missing values remaining in test:
Pclass        0
Sex           0
Age           0
Fare          0
Embarked      0
Title         0
FamilySize    0
IsAlone       0
dtype: int64


In [10]:
# ============================================================
# STEP 2: MODEL TRAINING & EVALUATION (VERSION 2)
# ============================================================

# 1. Expanded Feature Set: 8 engineered & cleaned features
feature_v2 = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']

# 2. One-Hot Encoding: Convert categoricals (Sex, Embarked, Title) to binary columns
X_v2 = pd.get_dummies(train[feature_v2], drop_first=True)
y_v2 = train['Survived']
X_test_v2 = pd.get_dummies(test[feature_v2], drop_first=True)

# 3. Align Columns: Ensure test set has the exact same columns and ordering as train
X_v2, X_test_v2 = X_v2.align(X_test_v2, join='left', axis=1, fill_value=0)

# 4. Train/Validation Split (80% train / 20% validation) with random_state=42 for reproducibility
X_train_v2, X_val_v2, y_train_v2, y_val_v2 = train_test_split(X_v2, y_v2, test_size=0.2, random_state=42)

# 5. Train Random Forest (100 trees, max depth 5 to prevent overfitting)
model_v2 = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_v2.fit(X_train_v2, y_train_v2)

# 6. Evaluate on Unseen 20% Validation Data
val_preds_v2 = model_v2.predict(X_val_v2)
acc_v2 = accuracy_score(y_val_v2, val_preds_v2)

print(f"Old Validation Accuracy: 78.77%")
print(f"New Validation Accuracy: {acc_v2 * 100:.2f}% (Gain: +{acc_v2 * 100 - 78.77:.2f}%)")

# 7. Generate Upgraded Submission File for Kaggle
# test_preds_v2 = model_v2.predict(X_test_v2)
# submission_v2 = pd.DataFrame({
#     'PassengerId': test['PassengerId'],
#     'Survived': test_preds_v2
# })
# submission_v2.to_csv('submission_v2.csv', index=False)
# print("\n'submission_v2.csv' successfully created! (Scored 0.78708 on Kaggle Leaderboard)")


Old Validation Accuracy: 78.77%
New Validation Accuracy: 82.12% (Gain: +3.35%)


## Version 3 — Deck + ensemble · 0.76794

CV said 83.84%, Kaggle said 0.76794. `Deck` is mostly missing (77% of passengers have no cabin)
and the decks that remain hold 4–94 people each.


In [11]:
# 1. Extract 'Deck' from the first letter of 'Cabin'
train['Deck'] = train['Cabin'].str[0].fillna('U')
test['Deck'] = test['Cabin'].str[0].fillna('U')

# Group rare decks ('T', 'G') into 'U'
train['Deck'] = train['Deck'].replace(['T', 'G'], 'U')
test['Deck'] = test['Deck'].replace(['T', 'G'], 'U')

# 2. Log transform Fare to normalize extreme ticket prices
train['LogFare'] = np.log1p(train['Fare'])
test['LogFare'] = np.log1p(test['Fare'])

# 3. Age * Class interaction feature (socio-economic priority index)
train['Age_Class'] = train['Age'] * train['Pclass']
test['Age_Class'] = test['Age'] * test['Pclass']

# Check survival rate by Deck!
print("--- Survival Rate by Deck ---")
print(train.groupby('Deck')['Survived'].agg(['count', 'mean']).sort_values(by='mean', ascending=False))


--- Survival Rate by Deck ---
      count      mean
Deck                 
D        33  0.757576
E        32  0.750000
B        47  0.744681
F        13  0.615385
C        59  0.593220
A        15  0.466667
U       692  0.300578


In [12]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 1. Select the Version 3 feature set
features_v3 = ['Pclass', 'Sex', 'Age', 'LogFare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'Deck', 'Age_Class']

# 2. One-hot encode categoricals
X_v3 = pd.get_dummies(train[features_v3], drop_first=True)
y_v3 = train['Survived']
X_test_v3 = pd.get_dummies(test[features_v3], drop_first=True)

# 3. Align train and test column structures
X_v3, X_test_v3 = X_v3.align(X_test_v3, join='left', axis=1, fill_value=0)

# 4. Define our two diverse models
rf_model = RandomForestClassifier(n_estimators=150, max_depth=5, min_samples_split=4, random_state=42)
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    hgb_model = HistGradientBoostingClassifier(
        max_iter=100, max_depth=4, learning_rate=0.05, random_state=42)
except ImportError:
    # sklearn < 0.21 (older Kaggle images) has no HistGradientBoosting.
    from sklearn.ensemble import GradientBoostingClassifier
    hgb_model = GradientBoostingClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)

# 5. Build an Ensemble (Voting Classifier combines both model votes)
ensemble_model = VotingClassifier(
    estimators=[('rf', rf_model), ('hgb', hgb_model)],
    voting='soft'
)

# 6. Evaluate with 5-Fold Stratified Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(ensemble_model, X_v3, y_v3, cv=cv, scoring='accuracy')

print(f"5-Fold CV Accuracy: {cv_scores.mean() * 100:.2f}% (+/- {cv_scores.std() * 100:.2f}%)")

# 7. Train on 100% of the training data for final submission!
# ensemble_model.fit(X_v3, y_v3)

# 8. Generate submission_v3.csv
# test_preds_v3 = ensemble_model.predict(X_test_v3)
# submission_v3 = pd.DataFrame({
#     'PassengerId': test['PassengerId'],
#     'Survived': test_preds_v3
# })
# submission_v3.to_csv('submission_v3.csv', index=False)

# print("\n'submission_v3.csv' created successfully using 100% trained ensemble!")
# print(f"File shape: {submission_v3.shape}")
# print(submission_v3.head())


5-Fold CV Accuracy: 84.29% (+/- 2.88%)


## Version 4 — Pruned + regularised · 0.77272

Drop `Deck` and `Age_Class`, add leaf regularisation, train on all 891 rows.


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# ============================================================
# VERSION 4: PRUNED & REGULARIZED CHAMPION (100% TRAINING)
# ============================================================

# 1. Back to our Champion clean features (no noisy Deck or Age_Class)
features_v4 = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']

# 2. One-hot encoding
X_v4 = pd.get_dummies(train[features_v4], drop_first=True)
y_v4 = train['Survived']
X_test_v4 = pd.get_dummies(test[features_v4], drop_first=True)

# 3. Column Alignment
X_v4, X_test_v4 = X_v4.align(X_test_v4, join='left', axis=1, fill_value=0)

# 4. Initialize Regularized Random Forest:
#    - max_depth=5 (prevents deep tree memorization)
#    - min_samples_split=6 (needs at least 6 samples before creating a new branch)
#    - min_samples_leaf=4 (strictly forbids single-leaf passenger memorization)
model_v4 = RandomForestClassifier(
    n_estimators=150,
    max_depth=5,
    min_samples_split=6,
    min_samples_leaf=4,
    random_state=42
)

# 5. Evaluate with 5-Fold Stratified Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_v4 = cross_val_score(model_v4, X_v4, y_v4, cv=cv, scoring='accuracy')

print(f"Version 4 CV Accuracy: {scores_v4.mean() * 100:.2f}% (+/- {scores_v4.std() * 100:.2f}%)")

# 6. Fit on 100% of training data (all 891 passengers)
# model_v4.fit(X_v4, y_v4)

# 7. Generate submission_v4.csv
# test_preds_v4 = model_v4.predict(X_test_v4)
# submission_v4 = pd.DataFrame({
#     'PassengerId': test['PassengerId'],
#     'Survived': test_preds_v4
# })
# submission_v4.to_csv('submission_v4.csv', index=False)

# print("\n'submission_v4.csv' created successfully!")
# print(f"Differences from Version 2: {(submission_v2['Survived'] != submission_v4['Survived']).sum()} passengers")
# print(submission_v4.head())


Version 4 CV Accuracy: 83.17% (+/- 2.54%)


## Version 5 — Family-group survival · 0.81100

Passengers evacuated in groups, and groups are split across train and test. `FamilySurvival` is
1.0 / 0.0 / 0.5 from the *other* members of a passenger's surname+fare or ticket group.
`.dropna()` keeps it clean — test labels are `NaN`, so they're never read.


In [14]:
# ============================================================
# VERSION 5: FAMILY & TICKET GROUP SURVIVAL (THE WCG MODEL)
# ============================================================

# 1. Combine train and test temporarily to link families across both sets
all_data = pd.concat([train, test], sort=False).reset_index(drop=True)

# 2. Extract Surnames and create unique Family Groups (Surname + Fare)
all_data['Surname'] = all_data['Name'].apply(lambda x: x.split(',')[0].strip())
all_data['FamilyGroup'] = all_data['Surname'] + '_' + all_data['Fare'].astype(str)

# 3. Default neutral family survival
all_data['FamilySurvival'] = 0.5

# 4. Calculate FamilySurvival strictly from OTHER family or shared-ticket members
for idx, row in all_data.iterrows():
    # Check family members with same Surname + Fare
    grp_mask = (all_data['FamilyGroup'] == row['FamilyGroup']) & (all_data.index != idx)
    grp_survivors = all_data.loc[grp_mask, 'Survived'].dropna()
    
    # Check shared ticket travelers (friends/nannies traveling together)
    tkt_mask = (all_data['Ticket'] == row['Ticket']) & (all_data.index != idx)
    tkt_survivors = all_data.loc[tkt_mask, 'Survived'].dropna()
    
    other_members = pd.concat([grp_survivors, tkt_survivors])
    
    if len(other_members) > 0:
        if other_members.max() == 1.0:
            all_data.loc[idx, 'FamilySurvival'] = 1.0
        elif other_members.min() == 0.0:
            all_data.loc[idx, 'FamilySurvival'] = 0.0

print("FamilySurvival distribution across all passengers:")
print(all_data['FamilySurvival'].value_counts())

# 5. Split back into train and test sets
train_v5 = all_data.iloc[:len(train)].copy()
test_v5 = all_data.iloc[len(train):].copy()

# 6. Prepare Feature Matrix
features_v5 = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'FamilySurvival']

X_v5 = pd.get_dummies(train_v5[features_v5], drop_first=True)
y_v5 = train_v5['Survived'].astype(int)
X_test_v5 = pd.get_dummies(test_v5[features_v5], drop_first=True)
X_v5, X_test_v5 = X_v5.align(X_test_v5, join='left', axis=1, fill_value=0)

# 7. Cross-Validation Check
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_v5 = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
scores_v5 = cross_val_score(rf_v5, X_v5, y_v5, cv=cv, scoring='accuracy')

print(f"\nVersion 5 CV Accuracy: {scores_v5.mean() * 100:.2f}% (+/- {scores_v5.std() * 100:.2f}%)")

# 8. Fit and Export submission.csv
rf_v5.fit(X_v5, y_v5)
test_preds_v5 = rf_v5.predict(X_test_v5)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_preds_v5
})
submission.to_csv('submission.csv', index=False)

print("\n'submission.csv' successfully created for Version 5!")
print(submission.head())


FamilySurvival distribution across all passengers:
0.5    763
1.0    303
0.0    243
Name: FamilySurvival, dtype: int64

Version 5 CV Accuracy: 84.51% (+/- 3.04%)

'submission.csv' successfully created for Version 5!
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1


## Next

- `fare_per_person` — `Fare` is the whole-ticket price, not one berth
- Apply the group signal to women and children only
- Repeated CV (5×5) instead of a single split
